# What Makes an Airbnb Worth the Price?
## Predicting Nightly Listing Prices Across New York City

---

This notebook builds regression models to predict the nightly price of Airbnb listings in New York City:

1. **Data Loading & Cleaning** — Inside Airbnb data, outlier filtering, and imputation
2. **Exploratory Visualizations** — price distributions and feature relationships
3. **Predictive Modeling** — naive baseline, Ridge regression, and tuned Random Forest
4. **Geographic Visualization** — median prices across NYC neighborhoods

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import json
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

DATA_DIR = Path(".").resolve()
LISTINGS_PATH = DATA_DIR / "listings.csv.gz"
GEOJSON_PATH = DATA_DIR / "neighbourhoods.geojson"


## 1. Data Loading

The dataset comes from [Inside Airbnb](https://insideairbnb.com/get-the-data/), an independent project that scrapes and publishes Airbnb listing data. The NYC file contains approximately 50,000+ listings with ~75 columns covering pricing, location, property details, host information and reviews.

In [ ]:
df = pd.read_csv(LISTINGS_PATH)
print(f"Loaded {len(df):,} listings from {LISTINGS_PATH.name}")

In [ ]:
df.head(10)

In [ ]:
# Check the dimensions of the dataset (rows, columns)
print(df.shape)

In [ ]:
df.info()

## 2. Feature Selection & Cleaning

We narrow the dataset to the columns most relevant to price prediction, then perform cleaning:

- **Price**: stored as a string like `"$120.00"` — strip `$` and commas and cast to float
- **Bathrooms**: extract the numeric value from text like `"1.5 baths"`
- **Outliers**: keep listings with nightly price between **$10 and $2,000**
- **Missing values**: impute where reasonable instead of dropping ~60% of rows (see audit below)

`neighbourhood_group_cleansed`, `latitude`, and `longitude` are kept for EDA but **dropped before modeling** — neighborhood dummies capture location; borough is redundant given neighborhood.

In [ ]:
# neighbourhood_group_cleansed, latitude, longitude are included here
# so Visualizations 3 and 4 still work.
# They will be DROPPED right before encoding (see Section 4).

features = [
    'price', 
    'neighbourhood_cleansed',
    'neighbourhood_group_cleansed',  # kept for EDA only — dropped before modeling
    'room_type', 
    'accommodates', 
    'bedrooms', 
    'bathrooms_text',
    'host_is_superhost', 
    'number_of_reviews', 
    'review_scores_rating', 
    'minimum_nights', 
    'latitude',    # kept for EDA only — dropped before modeling
    'longitude'    # kept for EDA only — dropped before modeling
]

df_clean = df[features].copy()

In [ ]:
# Clean the price column — remove $ and commas, convert to float
if not pd.api.types.is_numeric_dtype(df_clean['price']):
    df_clean['price'] = (
        df_clean['price']
        .astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .pipe(pd.to_numeric, errors='coerce')
    )

# Extract the number from the bathrooms text field (e.g. '1.5 baths' -> 1.5)
df_clean['bathrooms'] = df_clean['bathrooms_text'].str.extract(r'([0-9.]+)', expand=False).astype(float)
df_clean = df_clean.drop('bathrooms_text', axis=1)

# Require a listed price, then filter unrealistic nightly rates
rows_before_price = len(df_clean)
df_clean = df_clean.dropna(subset=['price'])
df_clean = df_clean[(df_clean['price'] >= 10) & (df_clean['price'] <= 2000)].copy()
print(f"Rows with valid price ($10–$2,000): {len(df_clean):,} (dropped {rows_before_price - len(df_clean):,})")

In [ ]:
# Audit missing values BEFORE imputation
print("Missing values per column (before imputation):")
print(df_clean.isnull().sum().sort_values(ascending=False))
print(f"\nRows before imputation: {len(df_clean):,}")

# Impute numeric features by room type where it makes sense
for col in ['bedrooms', 'bathrooms']:
    median_by_room = df_clean.groupby('room_type')[col].transform('median')
    global_median = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_by_room).fillna(global_median)

# Review score: only meaningful when there are reviews; otherwise use global median
review_median = df_clean.loc[df_clean['number_of_reviews'] > 0, 'review_scores_rating'].median()
df_clean['review_scores_rating'] = df_clean['review_scores_rating'].fillna(review_median)

# Superhost: treat missing as its own category (often newer hosts)
df_clean['host_is_superhost'] = df_clean['host_is_superhost'].fillna('unknown')

print(f"\nRows after imputation: {len(df_clean):,}")
print(f"Rows retained vs raw upload: {len(df_clean)/len(df)*100:.1f}%")

**Takeaway:** Listings without a price are excluded. Other fields are imputed so we retain far more listings than a strict `dropna()` (~15k vs ~20k+ with price). Review scores for zero-review listings use the cohort median — a simplification that favors established listings; interpret coefficients accordingly.


In [ ]:
df_clean.info()

## 3. Exploratory Data Analysis

Before modeling, we visualize the raw data to understand distributions, spot outliers, and identify which features are most related to price.

### Visualization 1 — Price Distribution

The histogram below shows the raw distribution of nightly prices. The X-axis is capped at $1,000 to reveal the shape of the bulk of listings — a small number of luxury outliers would otherwise collapse the chart into a flat line.

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 1: Price Distribution Histogram
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['price'], bins=100, kde=True, color='blue')
plt.title('Distribution of Airbnb Nightly Prices in NYC', fontsize=14)
plt.xlabel('Price (USD)', fontsize=12)
plt.ylabel('Number of Listings', fontsize=12)
plt.xlim(0, 1000)
plt.show()

**Takeaway:** The distribution is strongly right-skewed — most listings cluster between $50–$200/night, with a long tail of expensive outliers. This skewness confirms that we should log-transform the price before modeling so the regression isn't disproportionately influenced by luxury listings.

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 2: Boxplot of Price by Room Type
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.boxplot(x='room_type', y='price', data=df_clean, palette='Set2')
plt.title('Nightly Price Spread by Room Type', fontsize=14)
plt.xlabel('Room Type', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.ylim(0, 500)
plt.show()

**Takeaway:** Entire homes command significantly higher prices and wider variance — a penthouse and a studio are both "entire homes." Private rooms are consistently cheaper. Shared rooms are the most affordable but also the rarest listing type in NYC.

### Visualization 3 — Price by Borough

Manhattan and Brooklyn dominate NYC's Airbnb market. This boxplot compares median prices and spread across all five boroughs, sorted from most to least expensive.

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 3: Boxplot of Price by Borough
# neighbourhood_group_cleansed is used here for EDA —
# it will be dropped before modeling in Section 4.
# ---------------------------------------------------------
plt.figure(figsize=(12, 6))
borough_order = (df_clean
    .groupby('neighbourhood_group_cleansed')['price']
    .median()
    .sort_values(ascending=False)
    .index)
sns.boxplot(x='neighbourhood_group_cleansed', y='price', data=df_clean,
            order=borough_order, palette='Set3')
plt.title('Nightly Price Spread by Borough', fontsize=14)
plt.xlabel('Borough', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.ylim(0, 500)
plt.show()

**Takeaway:** Manhattan has the highest median price by a clear margin, followed by Brooklyn. The Bronx and Staten Island have lower medians but also far fewer listings. Queens sits in the middle — likely driven by proximity to JFK/LGA for budget travelers.

### Visualization 4 — Correlation Heatmap

This heatmap shows Pearson correlation coefficients between price and all numeric features. Values close to +1 indicate a strong positive relationship with price; values near 0 indicate little **linear** relationship — but this does not mean the feature is unimportant.

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 4: Correlation Heatmap
# ---------------------------------------------------------
numeric_cols = ['price', 'accommodates', 'bedrooms', 'bathrooms',
                'number_of_reviews', 'review_scores_rating', 'minimum_nights']
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Correlation Between Numeric Features and Price', fontsize=14)
plt.tight_layout()
plt.show()

**Takeaway:** `accommodates`, `bedrooms`, and `bathrooms` show the strongest positive correlations with price — larger listings cost more, unsurprisingly. `number_of_reviews` has a slight negative correlation, suggesting that cheaper, high-turnover listings accumulate more reviews.

`review_scores_rating` shows almost no linear correlation with price here, but this does **not** mean it is unimportant — it means its relationship with price is non-linear or conditional on other features (e.g., a high rating may matter more for expensive entire-home listings than for budget shared rooms). The Random Forest is motivated by these kinds of interaction effects across all features, not by any single feature's linear correlation alone.

## 4. Preprocessing for Modeling

### Log-Transformation of Price

Because the price distribution is right-skewed, we apply a `log1p` transformation (log(1 + price)) before modeling. This compresses the scale, reduces the influence of luxury outliers, and makes the residuals more normally distributed — a key assumption of linear regression.

Visualization 5 shows that the log-transformed prices are approximately bell-shaped, confirming the transformation is appropriate.

In [ ]:
# ---------------------------------------------------------
# 1. Log-Transform the Target Variable (Price)
# ---------------------------------------------------------
df_clean['log_price'] = np.log1p(df_clean['price'])

# ---------------------------------------------------------
# VISUALIZATION 5: Log-Transformed Price Distribution
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['log_price'], bins=50, kde=True, color='green')
plt.title('Log-Transformed Price Distribution', fontsize=14)
plt.xlabel('Log(Price)', fontsize=12)
plt.show()

**Takeaway:** Log-transformed prices follow a near-normal distribution, with much less skew than the raw prices. The model will predict in log-space and we'll convert back to dollars for interpretation.

### One-Hot Encoding & Train/Test Split

Before encoding, we drop three columns that would cause problems in the model:
- **`neighbourhood_group_cleansed`** (borough) — redundant with `neighbourhood_cleansed`. Every neighborhood belongs to exactly one borough, so encoding both creates perfect collinearity.
- **`latitude` and `longitude`** — redundant with neighborhood dummies, and their relationship with price is non-linear (price doesn't increase uniformly as you move north), which Ridge regression cannot learn from raw coordinates.

Categorical columns are then converted to binary dummy variables. `drop_first=True` removes one dummy per group to avoid the multicollinearity that would otherwise break linear regression. The data is split 80/20 into training and test sets with `random_state=42` for reproducibility.

In [ ]:
# ---------------------------------------------------------
# neighbourhood_group_cleansed: borough is fully determined by neighborhood
# latitude/longitude: already captured by neighborhood dummies at finer grain
# ---------------------------------------------------------
cols_to_drop_before_modeling = ['neighbourhood_group_cleansed', 'latitude', 'longitude']
df_model = df_clean.drop(cols_to_drop_before_modeling, axis=1)

# ---------------------------------------------------------
# 2. One-Hot Encoding for Categorical Variables
# ---------------------------------------------------------
categorical_features = ['neighbourhood_cleansed', 'room_type', 'host_is_superhost']

# drop_first=True avoids the dummy variable trap (multicollinearity in linear regression)
df_encoded = pd.get_dummies(df_model, columns=categorical_features, drop_first=True)

# Drop original price — the model predicts log_price, not price directly
df_final = df_encoded.drop('price', axis=1)

print("Final dataset shape for modeling:", df_final.shape)

In [ ]:
# ---------------------------------------------------------
# Train / Test Split + naive baseline
# ---------------------------------------------------------
y = df_final['log_price']
X = df_final.drop('log_price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nbhd_cols = [c for c in X_train.columns if c.startswith('neighbourhood_cleansed_')]
train_nbhd = X_train[nbhd_cols].idxmax(axis=1).str.replace('neighbourhood_cleansed_', '', regex=False)
train_log_median = (
    pd.DataFrame({'nbhd': train_nbhd, 'log_price': y_train.values})
    .groupby('nbhd')['log_price']
    .median()
)
test_nbhd = X_test[nbhd_cols].idxmax(axis=1).str.replace('neighbourhood_cleansed_', '', regex=False)
naive_log_pred = test_nbhd.map(train_log_median).fillna(y_train.median()).values

naive_dollars_pred = np.expm1(naive_log_pred)
y_test_dollars = np.expm1(y_test)
naive_rmse = np.sqrt(mean_squared_error(y_test_dollars, naive_dollars_pred))
naive_mae = mean_absolute_error(y_test_dollars, naive_dollars_pred)
naive_r2 = r2_score(y_test, naive_log_pred)

print("Naive baseline (train median by neighborhood):")
print(f"  Test R² (log scale): {naive_r2:.4f}")
print(f"  RMSE (dollars):      ${naive_rmse:.2f}/night")
print(f"  MAE (dollars):       ${naive_mae:.2f}/night")

## 5. Predictive Modeling

### Naive baseline — median price by neighborhood

We predict each test listing using the **median log price in its neighborhood**, using medians computed on the **training set only**.

### Model 1 — Ridge Regression

With ~200 neighborhood dummies, unregularized OLS is unstable. **Ridge regression** (`alpha=10`) handles multicollinearity better than plain linear regression.

### Model 2 — Random Forest Regressor

A non-linear ensemble with light hyperparameter search. We compare train vs test R², report **MAE and RMSE in dollars**, and run **5-fold CV on the training set only**.

In [ ]:
# ---------------------------------------------------------
# MODEL 1: Ridge Regression
# ---------------------------------------------------------
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train, y_train)
y_pred = ridge_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
y_pred_dollars = np.expm1(y_pred)
ridge_rmse_dollars = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))
ridge_mae_dollars = mean_absolute_error(y_test_dollars, y_pred_dollars)

print("Ridge Regression Results:")
print(f"  R-squared (R², log scale): {r2:.4f}")
print(f"  RMSE (log scale):          {rmse:.4f}")
print(f"  RMSE (dollars):            ${ridge_rmse_dollars:.2f}/night")
print(f"  MAE (dollars):             ${ridge_mae_dollars:.2f}/night")

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 6: Predicted vs. Actual — Ridge Regression
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Log Price', fontsize=12)
plt.ylabel('Predicted Log Price', fontsize=12)
plt.title('Ridge Regression: Predicted vs. Actual Price', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

**Takeaway:** The predicted vs. actual scatter plot shows that Ridge regression captures the general trend but struggles at the extremes — it under-predicts very cheap listings and over-predicts very expensive ones. The points fan out noticeably from the diagonal, suggesting the Ridge model is missing non-linear interactions (e.g., the value of being a Superhost probably isn't constant across all room types and neighborhoods).

### Model 2 — Random Forest Regressor

A Random Forest builds 100 independent decision trees and averages their predictions. Unlike linear regression, it can capture non-linear relationships and feature interactions without any manual feature engineering. We compare its R² and RMSE directly against the baseline.

We also report the **training R²** alongside the test R² — a large gap between the two would indicate overfitting (the model memorized training data rather than learning generalizable patterns).

In [ ]:
# ---------------------------------------------------------
# MODEL 2: Random Forest (hyperparameter search on train set)
# ---------------------------------------------------------
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [15, 25, None],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 0.3],
}
search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=12,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
)
search.fit(X_train, y_train)
rf_model = search.best_estimator_
print("Best RF params:", search.best_params_)

rf_y_pred = rf_model.predict(X_test)
rf_train_pred = rf_model.predict(X_train)
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_r2 = r2_score(y_test, rf_y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_y_pred))

rf_y_pred_dollars = np.expm1(rf_y_pred)
rf_rmse_dollars = np.sqrt(mean_squared_error(y_test_dollars, rf_y_pred_dollars))
rf_mae_dollars = mean_absolute_error(y_test_dollars, rf_y_pred_dollars)

print("\n--- MODEL COMPARISON ---")
print(f"  Naive (nbhd median)     R²: {naive_r2:.4f}")
print(f"  Ridge Regression        R²: {r2:.4f}")
print(f"  Random Forest (train)   R²: {rf_train_r2:.4f}")
print(f"  Random Forest (test)    R²: {rf_r2:.4f}")
print(f"  Overfitting gap:              {rf_train_r2 - rf_r2:.4f}")
print("-" * 40)
print(f"  Naive      RMSE/MAE ($): ${naive_rmse:.2f} / ${naive_mae:.2f}")
print(f"  Ridge      RMSE/MAE ($): ${ridge_rmse_dollars:.2f} / ${ridge_mae_dollars:.2f}")
print(f"  RF         RMSE/MAE ($): ${rf_rmse_dollars:.2f} / ${rf_mae_dollars:.2f}")

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 7: Predicted vs. Actual — Random Forest
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.scatter(y_test, rf_y_pred, alpha=0.3, color='seagreen', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Log Price', fontsize=12)
plt.ylabel('Predicted Log Price', fontsize=12)
plt.title('Random Forest: Predicted vs. Actual Price', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Dollar metrics reported above for all models.
print(f"Random Forest RMSE (log scale): {rf_rmse:.4f}")

**Takeaway:** The Random Forest substantially outperforms the Ridge regression on both metrics. The predicted vs. actual scatter clusters much more tightly around the diagonal. The overfitting gap between train and test R² tells us whether the model memorized the training data or learned genuine patterns — a gap under ~0.05 is healthy.

In [ ]:
# 5-fold CV on TRAINING data only (avoids test-set leakage)
cv_scores = cross_val_score(
    clone(rf_model),
    X_train,
    y_train,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)

print(f"5-Fold CV R² (train set only): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Individual folds: {[round(s, 4) for s in cv_scores]}")

### Visualization 8 — Feature Importance

The Random Forest assigns an importance score to each feature based on how much it reduces prediction error across all 100 trees. This tells us which factors actually drive Airbnb prices in NYC.

Because we removed latitude, longitude, and the borough column before modeling, the feature importance chart now reflects **genuine listing characteristics** rather than raw geographic coordinates.

In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 8: Feature Importance (grouped neighborhoods)
# ---------------------------------------------------------
importances = rf_model.feature_importances_
feature_names = X.columns

def readable_feature(name):
    if name.startswith('neighbourhood_cleansed_'):
        return 'Neighborhood: ' + name.replace('neighbourhood_cleansed_', '')
    labels = {
        'accommodates': 'Guest Capacity',
        'minimum_nights': 'Minimum Nights',
        'room_type_Private room': 'Room Type: Private Room',
        'bathrooms': 'Number of Bathrooms',
        'number_of_reviews': 'Total Reviews',
        'room_type_Hotel room': 'Room Type: Hotel Room',
        'review_scores_rating': 'Review Rating',
        'bedrooms': 'Number of Bedrooms',
        'host_is_superhost_t': 'Superhost Status',
        'host_is_superhost_unknown': 'Superhost: Unknown',
        'room_type_Shared room': 'Room Type: Shared Room',
    }
    return labels.get(name, name)

raw_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
raw_df['Group'] = raw_df['Feature'].apply(
    lambda x: 'Neighborhood (combined)' if x.startswith('neighbourhood_cleansed_') else readable_feature(x)
)
grouped = raw_df.groupby('Group', as_index=False)['Importance'].sum()
grouped = grouped.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Group', data=grouped, palette='magma')
plt.title('Top 15 Drivers of Airbnb Prices in NYC (Random Forest)', fontsize=14)
plt.xlabel('Importance Score (neighborhoods summed)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

**Takeaway:** With geographic redundancy removed, listing size (`accommodates`, `bedrooms`, `bathrooms`) and specific neighborhoods emerge as the clearest price drivers. Room type carries a large price effect: being a private room vs. an entire home is a significant penalty. Superhost status and review rating have smaller but non-zero importance — their effect is conditional on listing type and neighborhood, which is why it didn't appear in the linear correlation heatmap earlier.

## 6. Geographic Visualization
### Visualization 9 — Choropleth Map: Median Price by Neighborhood

The interactive map below shades each NYC neighborhood by its median nightly Airbnb price. Darker red indicates higher median prices. Hover over any neighborhood to see its name and exact median price. Neighborhoods with no listings appear in grey.

In [ ]:
# VISUALIZATION 8b: Residuals in dollars (Random Forest)
residuals = y_test_dollars - rf_y_pred_dollars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuals, bins=50, kde=True, ax=axes[0], color='teal')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribution of Prediction Errors ($)')
axes[0].set_xlabel('Actual − Predicted ($)')
axes[1].scatter(rf_y_pred_dollars, residuals, alpha=0.25, s=8, color='teal')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residuals vs Predicted Price')
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 9: Choropleth Map — Median Price by Neighborhood
# ---------------------------------------------------------
neighborhood_median = (
    df_clean.groupby('neighbourhood_cleansed')['price']
    .median()
    .reset_index()
    .rename(columns={'price': 'median_price'})
)

with open(GEOJSON_PATH, 'r') as f:
    nyc_geo = json.load(f)

geo_neighborhoods = {f['properties']['neighbourhood'] for f in nyc_geo['features']}
data_neighborhoods = set(neighborhood_median['neighbourhood_cleansed'])
matched = data_neighborhoods & geo_neighborhoods
print(f"Matched neighborhoods: {len(matched)} / {len(data_neighborhoods)} in data")
if data_neighborhoods - geo_neighborhoods:
    print("  In data only (sample):", sorted(data_neighborhoods - geo_neighborhoods)[:5])

m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=nyc_geo,
    name='Median Nightly Price',
    data=neighborhood_median,
    columns=['neighbourhood_cleansed', 'median_price'],
    key_on='feature.properties.neighbourhood',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Median Nightly Price (USD)',
    nan_fill_color='lightgrey',
    highlight=True,
).add_to(m)

neighborhood_dict = neighborhood_median.set_index('neighbourhood_cleansed')['median_price'].to_dict()

for feature in nyc_geo['features']:
    name = feature['properties']['neighbourhood']
    price = neighborhood_dict.get(name)
    feature['properties']['median_price'] = f"${price:.0f}" if price else "No data"

folium.GeoJson(
    nyc_geo,
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['neighbourhood', 'median_price'],
        aliases=['Neighborhood:', 'Median Price:'],
        localize=True,
        sticky=True,
    ),
).add_to(m)

folium.LayerControl().add_to(m)
out_path = DATA_DIR / 'nyc_airbnb_choropleth.html'
m.save(str(out_path))
print(f"Map saved to {out_path}")
m

## 7. Conclusion

In [ ]:
# Dynamic conclusion table
print("=" * 72)
print(f"{'Model':<36} {'R² (log)':>10} {'RMSE $':>10} {'MAE $':>10}")
print("-" * 72)
print(f"{'Naive (nbhd median)':<36} {naive_r2:>10.4f} {naive_rmse:>10.2f} {naive_mae:>10.2f}")
print(f"{'Ridge Regression':<36} {r2:>10.4f} {ridge_rmse_dollars:>10.2f} {ridge_mae_dollars:>10.2f}")
print(f"{'Random Forest (test)':<36} {rf_r2:>10.4f} {rf_rmse_dollars:>10.2f} {rf_mae_dollars:>10.2f}")
print(f"{'Random Forest (train)':<36} {rf_train_r2:>10.4f} {'—':>10} {'—':>10}")
print("=" * 72)
print(f"\nRandom Forest 5-Fold CV R² (train only): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Overfitting gap (train - test R²):         {rf_train_r2 - rf_r2:.4f}")

**Key findings:**
- Listing size (`accommodates`, `bedrooms`, `bathrooms`) and neighborhood drive most of the price signal.
- Random Forest beats Ridge and the naive neighborhood median on test R² and dollar RMSE/MAE.
- Imputation retains more listings than listwise deletion; review-score imputation introduces mild bias toward reviewed listings.

**Limitations:**
- Snapshot data (not forecasting); prices reflect one Inside Airbnb scrape.
- Models predict association, not causal “what if” effects of host choices.
- Luxury outliers above $2,000/night are excluded by design.